# Generate non-adiabatic steady flamelet for $NH_3$/$H_2$-air

***

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from spitfire.chemistry import flamelet
from spitfire.chemistry import tabulation
from spitfire.chemistry import analysis
from spitfire import Flamelet, FlameletSpec
from spitfire.chemistry.mechanism import ChemicalMechanismSpec
from cantera import gas_constant
from PCAfold import preprocess
from PCAfold import reduction
import cantera as ct
import plotly as plty
from plotly import express as px
from plotly import graph_objects as go
import time

%matplotlib inline

In [2]:
def plot_3d(x, y, z, color, cmap='plasma', s=2, xlabel='x', ylabel='y', zlabel='z'):

    fig = go.Figure(data=[go.Scatter3d(
        x=x.ravel(),
        y=y.ravel(),
        z=z.ravel(),
        mode='markers',
        marker=dict(size=s,
                    color=color.ravel(),
                    colorscale=cmap,
                    opacity=1,
                    colorbar=dict(thickness=20)))])
    
    fig.update_layout(autosize=False,
                      width=1000, height=600,
                      margin=dict(l=65, r=50, b=65, t=90),
                      scene = dict(xaxis_title=xlabel,
                                   yaxis_title=ylabel,
                                   zaxis_title=zlabel))
    
    fig.show()

In [3]:
ct.__version__

'2.6.0'

**Save data sets to `.csv`?**

In [4]:
save_csv = True

**Specify `filename_prefix` that will be added to the saved `.csv` files:**

In [5]:
filename_prefix = 'non-adiabatic-SLF-NH3-H2-air-25perc'

**Generate transient part?**

In [6]:
generate_transient = False

**Remove $N_2$ species when saving the data set?**

In [7]:
remove_N2 = False

#### Specify chemical mechanism:

In [8]:
chemical_mechanism = ChemicalMechanismSpec('ammonia-1atm.yaml', 'gas')

In [9]:
species_names = chemical_mechanism.species_names
print(species_names)

['AR', 'N2', 'HE', 'H2', 'H', 'O2', 'O', 'H2O', 'OH', 'H2O2', 'HO2', 'NO', 'N2O', 'NO2', 'HNO', 'HNO2', 'HONO', 'HONO2', 'N2H2', 'H2NN', 'NH2OH', 'HNOH', 'NH3', 'N2H4', 'N', 'NO3', 'NH', 'NNH', 'NH2', 'H2NO', 'N2H3']


#### Specify fuel and air streams:

In [10]:
pressure = 101325.

# Fuel and air streams:
fuel = chemical_mechanism.stream('X', 'NH3:3 H2:1')
fuel.TP = 300., pressure
air = chemical_mechanism.stream(stp_air=True)
air.TP = 300., pressure

# Print information about stoichiometric mixture fraction:
Z_stoich = chemical_mechanism.stoich_mixture_fraction(fuel, air)
print('Stoichiometric mixture fraction:')
print(Z_stoich)

Stoichiometric mixture fraction:
0.12373159044403234


***

## Generate steady flamelet library:

In [16]:
dissipation_rates = np.vstack((np.logspace(-2, 0.68, 100)[:,None], np.linspace(5.05,20.2,70)[:,None], np.logspace(1.31,1.5,60)[:,None])).ravel()

In [17]:
n_mf = 200
n_defect_st = 10

In [18]:
# dissipation_rates = np.array([10**-2, 10**-1.5, 10**-1, 10**-0.5, 10**0, 10**0.5, 10*1, 30])
# dissipation_rates

In [19]:
# dissipation_rates = np.logspace(-2, 1.4, 200)

In [20]:
n_chi = len(dissipation_rates)

In [ ]:
flamelet_specs = {'mech_spec': chemical_mechanism,
                  'oxy_stream': air,
                  'fuel_stream': fuel,
                  'grid_points': n_mf,
                  'grid_cluster_intensity': 3,
                  'grid_type': 'clustered',
                  'include_enthalpy_flux': False,
                  'include_variable_cp': False}

tic = time.perf_counter()

l_steady = tabulation.build_nonadiabatic_defect_steady_slfm_library(flamelet_specs,
                                                                    diss_rate_values=dissipation_rates,
                                                                    diss_rate_log_scaled=True,
                                                                    diss_rate_ref='stoichiometric',
                                                                    n_defect_st=n_defect_st,
                                                                    verbose=True)

toc = time.perf_counter()
print(f'Time it took: {(toc - tic)/60:0.1f} minutes.\n')

----------------------------------------------------------------------------------
building nonadiabatic (defect) SLFM library
----------------------------------------------------------------------------------
- mechanism: ammonia-1atm.yaml
- 31 species, 203 reactions
- stoichiometric mixture fraction: 0.124
----------------------------------------------------------------------------------
----------------------------------------------------------------------------------
building adiabatic SLFM library
----------------------------------------------------------------------------------
- mechanism: ammonia-1atm.yaml
- 31 species, 203 reactions
- stoichiometric mixture fraction: 0.124
----------------------------------------------------------------------------------
   1/ 230 (chi_stoich =  1.0e-02 1/s)  converged in   2.24 s, T_max = 2009.1
   2/ 230 (chi_stoich =  1.1e-02 1/s)  converged in   0.03 s, T_max = 2008.5
   3/ 230 (chi_stoich =  1.1e-02 1/s)  converged in   0.03 s, T_max = 20

In [ ]:
print('The last stoichiometric dissipation rate:')
print(l_steady.dissipation_rate_stoich_values[-1])

***

## Generate transient flamelet library:

In [ ]:
jump_chi = 1.4

In [ ]:
if generate_transient:
    flamelet_extinction = Flamelet(FlameletSpec(library_slice=l_steady[:, -1], stoich_dissipation_rate=l_steady.dissipation_rate_stoich_values[-1]*jump_chi))
    l_transient = flamelet_extinction.integrate_to_steady(write_log=True)

***

## Generate and save two training data sets (steady & transient):

##### Mixture fraction:

In [ ]:
mixture_fraction_steady = l_steady.mixture_fraction_grid.flatten()[:,None]
print(np.shape(mixture_fraction_steady))

In [ ]:
l_steady.mixture_fraction_grid.shape

In [ ]:
if generate_transient:
    mixture_fraction_transient = l_transient.mixture_fraction_grid.flatten()[:,None]
    print(np.shape(mixture_fraction_transient))

##### Enthalpy defect

In [ ]:
# enthalpy_defect_steady = l_steady['enthalpy_defect'].flatten()[:,None]
# print(np.shape(enthalpy_defect_steady))

In [ ]:
l_steady.enthalpy_defect_stoich_values

In [ ]:
enthalpy_defect_steady = np.ones((n_mf, n_chi, n_defect_st))

for i in range(0,n_defect_st):

    enthalpy_defect_steady[:,:,i] = enthalpy_defect_steady[:,:,i] * l_steady.enthalpy_defect_stoich_values[i]

enthalpy_defect_steady = enthalpy_defect_steady.flatten()[:,None]
print(np.shape(enthalpy_defect_steady))

##### Density (Cantera "`density`"):

In [ ]:
l_steady = analysis.compute_density(chemical_mechanism, l_steady) # kg/m3
density_steady = l_steady['density'].flatten()[:,None]
print(np.shape(density_steady))

In [ ]:
if generate_transient:
    l_transient = analysis.compute_density(chemical_mechanism, l_transient) # kg/m3
    density_transient = l_transient['density'].flatten()[:,None]
    print(np.shape(density_transient))

##### Dissipation rates:

In [ ]:
chi_steady = l_steady.dissipation_rate_stoich_grid.flatten()[:,None]
print(np.shape(chi_steady))

In [ ]:
if generate_transient:
    chi_transient = jump_chi*l_steady.dissipation_rate_stoich_values[-1]*np.ones_like(mixture_fraction_transient)
    print(np.shape(chi_transient))

##### Isobaric heat capacity (Cantera "`cp_mass`"):

In [ ]:
l_steady = analysis.compute_isobaric_specific_heat(chemical_mechanism, l_steady) # J/kg/K
isobaric_heat_capacity_steady = l_steady['heat capacity cp'].flatten()[:,None]
print(np.shape(isobaric_heat_capacity_steady))

In [ ]:
if generate_transient:
    l_transient = analysis.compute_isobaric_specific_heat(chemical_mechanism, l_transient) # J/kg/K
    isobaric_heat_capacity_transient = l_transient['heat capacity cp'].flatten()[:,None]
    print(np.shape(isobaric_heat_capacity_transient))

##### Species mass fractions:

In [ ]:
ct_sa_steady, l_shape_steady = analysis.get_ct_solution_array(chemical_mechanism, l_steady)
mass_fractions_steady = ct_sa_steady.Y # [-]
print(np.shape(mass_fractions_steady))

In [ ]:
if generate_transient:
    ct_sa_transient, l_shape_transient = analysis.get_ct_solution_array(chemical_mechanism, l_transient)
    mass_fractions_transient = ct_sa_transient.Y # [-]
    print(np.shape(mass_fractions_transient))

##### Temperature:

In [ ]:
temperature_steady = ct_sa_steady.T[:,None] # K
print(np.shape(temperature_steady))

In [ ]:
if generate_transient:
    temperature_transient = ct_sa_transient.T[:,None] # K
    print(np.shape(temperature_transient))

##### Species production rates:

In [ ]:
production_rates_steady = ct_sa_steady.net_production_rates * ct_sa_steady.molecular_weights # kmol/m3-s * kg/kmol = kg/m3-s
production_rates_over_density_steady = np.divide(production_rates_steady, density_steady) # 1/s
print(np.shape(production_rates_steady))
print(np.shape(production_rates_over_density_steady))

In [ ]:
if generate_transient:
    production_rates_transient = ct_sa_transient.net_production_rates * ct_sa_transient.molecular_weights # kmol/m3-s * kg/kmol = kg/m3-s
    production_rates_over_density_transient = np.divide(production_rates_transient, density_transient) # 1/s
    print(np.shape(production_rates_transient))
    print(np.shape(production_rates_over_density_transient))

##### Species enthalpies:

In [ ]:
species_enthalpies_steady = ct_sa_steady.standard_enthalpies_RT * temperature_steady * gas_constant / ct_sa_steady.molecular_weights # J/kg
print(np.shape(species_enthalpies_steady))

In [ ]:
if generate_transient:
    species_enthalpies_transient = ct_sa_transient.standard_enthalpies_RT * temperature_transient * gas_constant / ct_sa_transient.molecular_weights # J/kg
    print(np.shape(species_enthalpies_transient))

##### Temperature source term $S_T$:

In [ ]:
temperature_source_steady = - np.sum(production_rates_steady * species_enthalpies_steady, axis=1) / density_steady.ravel() / isobaric_heat_capacity_steady.ravel() # K/s
temperature_source_steady = temperature_source_steady[:,None]
print(np.shape(temperature_source_steady))

In [ ]:
if generate_transient:
    temperature_source_transient = - np.sum(production_rates_transient * species_enthalpies_transient, axis=1) / density_transient.ravel() / isobaric_heat_capacity_transient.ravel() # K/s
    temperature_source_transient = temperature_source_transient[:,None]
    print(np.shape(temperature_source_transient))

##### Heat release rate:

Other two ways you can compute HRR, both give the same thing:

```python
heat_release_rate = - np.sum(ct_sa.net_production_rates * ct_sa.partial_molar_enthalpies, axis=1)
heat_release_rate = - np.sum(ct_sa.net_rates_of_progress * ct_sa.delta_enthalpy, axis=1)
```

see [Cantera tutorial](https://cantera.org/tutorials/python-tutorial.html).

In [ ]:
heat_release_rate_steady = - np.sum(production_rates_steady * species_enthalpies_steady, axis=1)
heat_release_rate_steady = heat_release_rate_steady[:,None]
print(np.shape(heat_release_rate_steady))

In [ ]:
if generate_transient:
    heat_release_rate_transient = - np.sum(production_rates_transient * species_enthalpies_transient, axis=1)
    heat_release_rate_transient = heat_release_rate_transient[:,None]
    print(np.shape(heat_release_rate_transient))

##### Specific enthalpy:

In [ ]:
specific_enthalpy_steady = ct_sa_steady.enthalpy_mass
specific_enthalpy_steady = specific_enthalpy_steady[:,None]
print(np.shape(specific_enthalpy_steady))

In [ ]:
if generate_transient:
    specific_enthalpy_transient = ct_sa_transient.enthalpy_mass
    specific_enthalpy_transient = specific_enthalpy_transient[:,None]
    print(np.shape(specific_enthalpy_transient))

In [ ]:
reduction.plot_3d_manifold(mixture_fraction_steady, 
                           np.log10(chi_steady), 
                           enthalpy_defect_steady,
                           color=temperature_steady,
                           color_map='inferno');

In [ ]:
plot_3d(mixture_fraction_steady, 
        np.log10(chi_steady), 
        enthalpy_defect_steady, 
        temperature_steady, 
        xlabel=r'Mixture fraction [-]',
        ylabel='Diss. rate [1/s]',
        zlabel='Enthalpy defect stoich. [MJ]',
        cmap='plasma', 
        s=2)

In [ ]:
plot_3d(mixture_fraction_steady, enthalpy_defect_steady, mass_fractions_steady[:,12], temperature_steady, cmap='plasma', s=2)

***

## Save the data sets:

Prepare state-space and sources of state-space data sets that will be saved:

In [ ]:
if remove_N2:
    STEADY_state_space = np.hstack((temperature_steady, mass_fractions_steady[:,0:-1]))
    STEADY_state_space_sources = np.hstack((temperature_source_steady, production_rates_over_density_steady[:,0:-1]))
    print(np.shape(STEADY_state_space))
    print(np.shape(STEADY_state_space_sources))
else:
    STEADY_state_space = np.hstack((temperature_steady, mass_fractions_steady))
    STEADY_state_space_sources = np.hstack((temperature_source_steady, production_rates_over_density_steady))
    print(np.shape(STEADY_state_space))
    print(np.shape(STEADY_state_space_sources))

In [ ]:
if generate_transient:
    if remove_N2:
        TRANSIENT_state_space = np.hstack((temperature_transient, mass_fractions_transient[:,0:-1]))
        TRANSIENT_state_space_sources = np.hstack((temperature_source_transient, production_rates_over_density_transient[:,0:-1]))
        print(np.shape(TRANSIENT_state_space))
        print(np.shape(TRANSIENT_state_space_sources))
    else:
        TRANSIENT_state_space = np.hstack((temperature_transient, mass_fractions_transient))
        TRANSIENT_state_space_sources = np.hstack((temperature_source_transient, production_rates_over_density_transient))
        print(np.shape(TRANSIENT_state_space))
        print(np.shape(TRANSIENT_state_space_sources))

In [ ]:
if remove_N2:
    state_space_names = ['T'] + species_names[0:-1]
    print(state_space_names)
else:
    state_space_names = ['T'] + species_names
    print(state_space_names)

In [ ]:
if save_csv:
    
    # Species names:
    np.savetxt(filename_prefix + '-state-space-names.csv', (state_space_names), delimiter=',', fmt="%s")
    if generate_transient: np.savetxt('../data-sets/TRANSIENT-' + filename_prefix + '-state-space-names.csv', (state_space_names), delimiter=',', fmt="%s")
    
    # State-space:
    np.savetxt(filename_prefix + '-state-space.csv', (STEADY_state_space), delimiter=',', fmt='%.16e')
    if generate_transient: np.savetxt('../data-sets/TRANSIENT-' + filename_prefix + '-state-space.csv', (TRANSIENT_state_space), delimiter=',', fmt='%.16e')
    
    # Sources of state-space:
    np.savetxt(filename_prefix + '-state-space-sources.csv', (STEADY_state_space_sources), delimiter=',', fmt='%.16e')
    if generate_transient: np.savetxt('../data-sets/TRANSIENT-' + filename_prefix + '-state-space-sources.csv', (TRANSIENT_state_space_sources), delimiter=',', fmt='%.16e')
    
    # Mixture fraction:
    np.savetxt(filename_prefix + '-mixture-fraction.csv', (mixture_fraction_steady), delimiter=',', fmt='%.16e')
    if generate_transient: np.savetxt('../data-sets/TRANSIENT-' + filename_prefix + '-mixture-fraction.csv', (mixture_fraction_transient), delimiter=',', fmt='%.16e')
    
    # Dissipation rates:
    np.savetxt(filename_prefix + '-dissipation-rates.csv', (chi_steady), delimiter=',', fmt='%.16e')
    if generate_transient: np.savetxt('../data-sets/TRANSIENT-' + filename_prefix + '-dissipation-rates.csv', (chi_transient), delimiter=',', fmt='%.16e')
    
    # Heat release rate:
    np.savetxt(filename_prefix + '-heat-release-rate.csv', (heat_release_rate_steady), delimiter=',', fmt='%.16e')
    if generate_transient: np.savetxt('../data-sets/TRANSIENT-' + filename_prefix + '-heat-release-rate.csv', (heat_release_rate_transient), delimiter=',', fmt='%.16e')
    
    # Specific enthalpy:
    np.savetxt(filename_prefix + '-specific-enthalpy.csv', (specific_enthalpy_steady), delimiter=',', fmt='%.16e')
    if generate_transient: np.savetxt('../data-sets/TRANSIENT-' + filename_prefix + '-specific-enthalpy.csv', (specific_enthalpy_transient), delimiter=',', fmt='%.16e')
        
    # Enthalpy defect:
    np.savetxt(filename_prefix + '-enthalpy-defect.csv', (enthalpy_defect_steady), delimiter=',', fmt='%.16e')
    if generate_transient: np.savetxt('../data-sets/TRANSIENT-' + filename_prefix + '-enthalpy-defect.csv', (enthalpy_defect_transient), delimiter=',', fmt='%.16e')
    
    print('Data sets saved!')

***

## Visualize the data sets

Concatenate data sets:

In [ ]:
if generate_transient:
    mixture_fractions = np.vstack((mixture_fraction_steady, mixture_fraction_transient))
    state_spaces = np.vstack((STEADY_state_space, TRANSIENT_state_space))
    state_spaces_sources = np.vstack((STEADY_state_space_sources, TRANSIENT_state_space_sources))
    chis = np.vstack((chi_steady, chi_transient))
    hrrs = np.vstack((heat_release_rate_steady, heat_release_rate_transient))
    h = np.vstack((specific_enthalpy_steady, specific_enthalpy_transient))
else:
    mixture_fractions = mixture_fraction_steady
    state_spaces = STEADY_state_space
    state_spaces_sources = STEADY_state_space_sources
    chis = chi_steady
    hrrs = heat_release_rate_steady
    h = specific_enthalpy_steady

In [ ]:
x_label = 'Mixture fraction [-]'
y_label = '$T$ [K]'
figure_size = (8,5)

In [ ]:
plt_both = reduction.plot_2d_manifold(mixture_fractions, state_spaces[:,0], color=chis.ravel(), x_label=x_label, y_label=y_label, colorbar_label='$\chi_{st}$ [1/s]', figure_size=figure_size)

In [ ]:
plt_both = reduction.plot_2d_manifold(mixture_fractions, state_spaces[:,0], color=hrrs.ravel(), x_label=x_label, y_label=y_label, colorbar_label='HRR [$W/m^3$]', figure_size=figure_size)

In [ ]:
select_variable = 6
plt_both = reduction.plot_2d_manifold(mixture_fractions, state_spaces[:,0], color=state_spaces[:,select_variable], x_label=x_label, y_label=y_label, colorbar_label='$Y_{' + state_space_names[select_variable] + '}$ [-]', figure_size=figure_size)

In [ ]:
select_source = 6
plt_both = reduction.plot_2d_manifold(mixture_fractions, state_spaces[:,0], color=state_spaces_sources[:,select_source], x_label=x_label, y_label=y_label, colorbar_label='$S_{' + state_space_names[select_source] + '}$ [-]', figure_size=figure_size)

In [ ]:
(_, idx_removed, idx_retained) = preprocess.remove_constant_vars(state_spaces)
pca = reduction.PCA(state_spaces[:,idx_retained], scaling='-1to1', n_components=3)
PCs = pca.transform(state_spaces[:,idx_retained])
PC_sources = pca.transform(state_spaces_sources[:,idx_retained], nocenter=True)

In [ ]:
plt_both = reduction.plot_2d_manifold(PCs[:,0],
                                      PCs[:,1],
                                      color=PC_sources[:,0],
                                      x_label=x_label,
                                      y_label=y_label,
                                      colorbar_label='$S_{Z, 1}$ [1/s]',
                                      color_map='inferno',
                                      figure_size=(5,5))

In [ ]:
plot_3d(PCs[:,0], PCs[:,1], PCs[:,2], PC_sources[:,0], cmap='inferno', s=2)

***